In [1]:
from google.cloud import bigquery

client = bigquery.Client(project="funnel-analysis-project-503810")

query = """
    SELECT
    traffic_source.medium AS channel,
    CASE
        WHEN (SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_number') = 1
        THEN 'new'
        ELSE 'returning'
    END AS user_type,
    COUNT(DISTINCT CASE WHEN event_name = 'page_view' THEN user_pseudo_id END) AS page_view_users,
    COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_pseudo_id END) AS purchase_users
    FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
    WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20201130'
    GROUP BY channel, user_type
    ORDER BY channel, user_type
"""

df = client.query(query).to_dataframe()

print(df.shape)
df.head(50)

C:\Users\dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(12, 4)


,channel,user_type,page_view_users,purchase_users
0,(data deleted),new,263,4
1,(data deleted),returning,4767,234
2,(none),new,17305,152
3,(none),returning,4832,197
4,<Other>,new,13046,111


,channel,user_type,page_view_users,purchase_users,conversion_rate,segment
0,(data deleted),new,263,4,0.015209,(data deleted)-new
1,(data deleted),returning,4767,234,0.049087,(data deleted)-returning
2,(none),new,17305,152,0.008784,(none)-new
3,(none),returning,4832,197,0.040770,(none)-returning
4,<Other>,new,13046,111,0.008508,<Other>-new
5,<Other>,returning,1326,65,0.049020,<Other>-returning
6,cpc,new,3951,35,0.008859,cpc-new
7,cpc,returning,352,17,0.048295,cpc-returning
8,organic,new,27863,217,0.007788,organic-new
9,organic,returning,5002,210,0.041983,organic-returning


In [61]:
df['conversion_rate'] = df['purchase_users']/df['page_view_users']

agg_df = df.groupby('channel')[['page_view_users', 'purchase_users']].sum().reset_index()
agg_df['conversion_rate'] = agg_df['purchase_users'].astype(float) / agg_df['page_view_users'].astype(float)
agg_df

,channel,page_view_users,purchase_users,conversion_rate
0,(data deleted),5030,238,0.047316
1,(none),22137,349,0.015765
2,<Other>,14372,176,0.012246
3,cpc,4303,52,0.012085
4,organic,32865,427,0.012993
5,referral,16117,359,0.022275


In [62]:
import pandas as pd

In [ ]:
agg_df['user_type'] = 'aggregate'



combined = pd.concat([df, agg_df], ignore_index=True)

In [69]:
agg_df.head(10)

,channel,page_view_users,purchase_users,conversion_rate,user_type
0,(data deleted),5030,238,0.047316,aggregate
1,(none),22137,349,0.015765,aggregate
2,<Other>,14372,176,0.012246,aggregate
3,cpc,4303,52,0.012085,aggregate
4,organic,32865,427,0.012993,aggregate
5,referral,16117,359,0.022275,aggregate


In [43]:
agg_df.dtypes

channel                str
page_view_users      Int64
purchase_users       Int64
conversion_rate    float64
user_type              str
dtype: object

In [67]:
combined

,channel,user_type,page_view_users,purchase_users,conversion_rate,segment
0,(data deleted),new,263,4,0.015209,(data deleted)-new
1,(data deleted),returning,4767,234,0.049087,(data deleted)-returning
2,(none),new,17305,152,0.008784,(none)-new
3,(none),returning,4832,197,0.04077,(none)-returning
4,<Other>,new,13046,111,0.008508,<Other>-new
5,<Other>,returning,1326,65,0.04902,<Other>-returning
6,cpc,new,3951,35,0.008859,cpc-new
7,cpc,returning,352,17,0.048295,cpc-returning
8,organic,new,27863,217,0.007788,organic-new
9,organic,returning,5002,210,0.041983,organic-returning


In [47]:
import plotly.express as px

# Optional: filter to just cpc and organic for the clearest comparison
plot_df = combined[combined['channel'].isin(['cpc', 'organic'])]

fig = px.bar(
    plot_df,
    x='channel',
    y='conversion_rate',
    color='user_type',
    barmode='group',
    title='Conversion Rate by Channel: Aggregate vs New vs Returning',
    labels={'conversion_rate': 'Conversion Rate', 'channel': 'Channel', 'user_type': 'Segment'}
)

fig.update_layout(yaxis_tickformat='.1%')
fig.show()

Simpson's Paradox Alert: CPC wins in both New (+0.1%) and Returning (+0.63%) segments. However, Organic's overall rate looks higher because 40% of Organic traffic consists of high-converting Returning users, compared to only 8% for CPC.

In [53]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "1. Conversion Rate by Segment",
        "2. Traffic Share Confounder",
        "3. Reweighted: What-If Same Mix"
    )
)

# Panel 1: Rates by segment
fig.add_trace(go.Bar(name='CPC Rates', x=['New', 'Returning', 'Aggregate'], y=[0.88, 4.81, 1.20],
                      marker_color='#636EFA', showlegend=True), row=1, col=1)
fig.add_trace(go.Bar(name='Organic Rates', x=['New', 'Returning', 'Aggregate'], y=[0.78, 4.18, 1.28],
                      marker_color='#EF553B', showlegend=True), row=1, col=1)

# Panel 2: Traffic mix per channel
fig.add_trace(go.Bar(name='New Mix %', x=['CPC', 'Organic'], y=[92, 60],
                      marker_color='#00CC96', showlegend=True), row=1, col=2)
fig.add_trace(go.Bar(name='Returning Mix %', x=['CPC', 'Organic'], y=[8, 40],
                      marker_color='#AB63FA', showlegend=True), row=1, col=2)

# Panel 3: Actual aggregate vs. reweighted (same-mix) aggregate
fig.add_trace(go.Bar(name='Actual (own mix)', x=['CPC', 'Organic'], y=[1.20, 1.28],
                      marker_color='#636EFA', showlegend=False), row=1, col=3)
fig.add_trace(go.Bar(name='Hypothetical (other channel\'s mix)', x=['CPC', 'Organic'], y=[2.45, 1.05],
                      marker_color='#EF553B', showlegend=False), row=1, col=3)

fig.update_layout(
    barmode='group',
    title_text="Demonstrating Simpson's Paradox: Channel Conversion",
    annotations=[
        dict(text="1. Conversion Rate by Segment", x=0.10, y=1.12, xref='paper', yref='paper', showarrow=False, font=dict(size=13)),
        dict(text="2. Traffic Share Confounder", x=0.50, y=1.12, xref='paper', yref='paper', showarrow=False, font=dict(size=13)),
        dict(text="3. Reweighted: What-If Same Mix", x=0.90, y=1.12, xref='paper', yref='paper', showarrow=False, font=dict(size=13)),
    ]
)
fig.update_yaxes(title_text="Conversion Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="% of Channel's Traffic", row=1, col=2)
fig.update_yaxes(title_text="Conversion Rate (%)", row=1, col=3)

fig.show()

In panel 3, once traffic mix is held constant (standardized) as the confounder, CPC's conversion rate is actually higher than Organic's — the reverse of what the raw aggregate showed. This confirms that traffic mix (the proportion of new vs. returning users) was the hidden variable dragging CPC's aggregate rate down, not any real difference in how well CPC converts visitors.